Step 1 — Bronze: read the raw CSV

# Robinhood Medallion Pipeline
Source: Robinhood Markets Consolidated Transactions (1099 tax file)
Layers: Bronze → Silver → Gold
Goal: Calculate profit/loss per trade from raw tax data

In [0]:
from pyspark.sql.functions import col, round, sum as spark_sum

In [0]:
df_raw = spark.read \
    .option("header", "true") \
    .csv("/Volumes/workspace/robinhood/raw_data/Robinhood Markets Consolidated Transactions.csv")


df_raw.show(5)
df_raw.printSchema()

Save as bronze_raw Delta table 

In [0]:
df_raw.write \
    .format("delta") \
    .mode("overwrite") \
    .option("delta.columnMapping.mode", "name") \
    .saveAsTable("workspace.robinhood.bronze_raw")

Step 2 — Silver: isolate the 1099-B rows

In [0]:

import io
import csv

# Read file as text lines
lines = spark.read.text("/Volumes/workspace/robinhood/raw_data/Robinhood Markets Consolidated Transactions.csv")
all_lines = [row.value for row in lines.collect()]

# Separate 1099-B header and data rows
header_1099b = None
data_1099b = []

for line in all_lines:
    if line.startswith("1099-B,ACCOUNT NUMBER"):
        header_1099b = line
    elif line.startswith("1099-B,") and "ACCOUNT NUMBER" not in line:
        data_1099b.append(line)

# Parse into proper DataFrame
def parse_line(line):
    reader = csv.reader(io.StringIO(line))
    return next(reader)

headers = parse_line(header_1099b)[1:]
rows = [parse_line(line)[1:] for line in data_1099b]

df_silver = spark.createDataFrame(rows, headers).select(
    "ACCOUNT NUMBER", "TAX YEAR", "DATE ACQUIRED",
    "SALE DATE", "DESCRIPTION", "SHARES",
    "COST BASIS", "SALES PRICE", "TERM"
)

df_silver.show(truncate=False)


In [0]:
# Drop rows where key numeric fields are null or empty
df_silver = df_silver.filter(
    col("COST BASIS").isNotNull() &
    col("SALES PRICE").isNotNull() &
    (col("COST BASIS") != "") &
    (col("SALES PRICE") != "")
)

print(f"Clean rows after null filter: {df_silver.count()}")

save it as a Delta table before moving to gold

In [0]:

df_silver = df_silver.toDF(
    "account_number", "tax_year", "date_acquired",
    "sale_date", "description", "shares",
    "cost_basis", "sales_price", "term"
)

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.robinhood.silver_trades")

In [0]:
df_silver.show()

Step 3 — Gold Prep: cast numeric columns

This separates:

* cleaning
* typing
* calculations

In [0]:

df_gold = df_silver \
    .withColumn("tax_year", col("tax_year").cast("int")) \
    .withColumn("shares", col("shares").cast("double")) \
    .withColumn("cost_basis", col("cost_basis").cast("double")) \
    .withColumn("sales_price", col("sales_price").cast("double"))

Step 4: — Gold: calculate profit/loss

In [0]:
df_gold_trades = df_gold.withColumn(
    "profit_loss",
    round(col("sales_price") - col("cost_basis"), 2)
)
df_gold_trades.show(5)

Save it as Delta table 

In [0]:
%sql
DROP TABLE IF EXISTS workspace.robinhood.gold_trades

In [0]:
df_gold_trades.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.robinhood.gold_trades")

Gold - Create the summary table

In [0]:
# Gold summary grouped by term (short vs long)
df_gold_summary = df_gold_trades.groupBy("term").agg(
    round(spark_sum("profit_loss"), 2).alias("total_profit_loss")
).orderBy("term")

df_gold_summary.show()

Save it as Delta table

In [0]:
%sql
DROP TABLE IF EXISTS workspace.robinhood.gold_summary

In [0]:
df_gold_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.robinhood.gold_summary")

In [0]:
%sql
SHOW TABLES IN workspace.robinhood